In [1]:
import os
path = '/kaggle/input/datasets/ekrasafdar'
print("Contents of /kaggle/input/datasets/ekrasafdar:")
for item in sorted(os.listdir(path)):
    full = os.path.join(path, item)
    if os.path.isdir(full):
        sub = sorted([s for s in os.listdir(full) if os.path.isdir(os.path.join(full, s))])
        print(f"  📁 {item}/  → subfolders: {sub}")

Contents of /kaggle/input/datasets/ekrasafdar:
  📁 brain-tumor-mri/  → subfolders: ['Testing', 'Training']
  📁 sartajbhuvajibrain-tumor-classification-mri/  → subfolders: ['Train', 'Val']


In [ ]:
import os, numpy as np, tensorflow as tf, cv2
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre_mob
from tensorflow.keras.applications.resnet50 import preprocess_input as pre_res
from tensorflow.keras.applications.efficientnet import preprocess_input as pre_eff
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

tf.get_logger().setLevel('ERROR')
np.random.seed(42)
tf.random.set_seed(42)

# ========== AUTO-DETECT PRIMARY DATASET ==========
# Search under /kaggle/input recursively for a folder with Training + Testing
PRIMARY_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    subdirs = [d for d in dirs if os.path.isdir(os.path.join(root, d))]
    has_train = any(x.lower() in ['training', 'train'] for x in subdirs)
    has_test = any(x.lower() in ['testing', 'test'] for x in subdirs)
    if has_train and has_test:
        # Verify it has 4 class subfolders inside Training
        train_candidates = [d for d in subdirs if d.lower() in ['training', 'train']]
        for tc in train_candidates:
            train_path = os.path.join(root, tc)
            try:
                classes = [c for c in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, c))]
                if len(classes) >= 4:
                    PRIMARY_DIR = root
                    print(f"✅ Primary dataset: {root}")
                    print(f"   Training subfolder: {tc}")
                    print(f"   Classes: {classes}")
                    break
            except:
                pass
        if PRIMARY_DIR:
            break

if not PRIMARY_DIR:
    raise FileNotFoundError("Primary dataset not found. Attach it via + Add Input.")

# Resolve exact subfolder names
subdirs = [d for d in os.listdir(PRIMARY_DIR) if os.path.isdir(os.path.join(PRIMARY_DIR, d))]
TRAIN_NAME = next((d for d in subdirs if d.lower() in ['training', 'train']), None)
TEST_NAME = next((d for d in subdirs if d.lower() in ['testing', 'test']), None)

TRAIN_DIR = os.path.join(PRIMARY_DIR, TRAIN_NAME)
TEST_DIR = os.path.join(PRIMARY_DIR, TEST_NAME)

IMG_SIZE, BATCH = (224, 224), 32
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

# ========== AUTO-FIND SARTAJ ==========
SARTAJ_DIR = None
SARTAJ_TEST = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'sartaj' in root.lower():
        subdirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
        if 'Val' in subdirs or 'val' in subdirs:
            SARTAJ_DIR = root
            SARTAJ_TEST = os.path.join(root, 'Val') if 'Val' in subdirs else os.path.join(root, 'val')
            print(f"✅ SARTAJ: {root}")
            break

# ========== AUTO-FIND FIGSHARE ==========
FIGSHARE_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    for d in dirs:
        path = os.path.join(root, d)
        try:
            sub = [s for s in os.listdir(path) if os.path.isdir(os.path.join(path, s))]
            sub_lower = [s.lower() for s in sub]
            if all(c in sub_lower for c in ['glioma', 'meningioma', 'pituitary']):
                if 'notumor' not in sub_lower and 'no_tumor' not in sub_lower:
                    FIGSHARE_DIR = path
                    print(f"✅ Figshare: {path}")
                    break
        except:
            pass
    if FIGSHARE_DIR:
        break

# ========== HELPERS ==========
def train_and_save(name, pre_fn, seed):
    tf.random.set_seed(seed); np.random.seed(seed)
    inp = layers.Input(shape=(224, 224, 3))
    if name == 'mobilenetv2':
        base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
    elif name == 'resnet50':
        base = ResNet50(weights='imagenet', include_top=False, input_tensor=inp)
    else:
        base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inp)
    
    base.trainable = True
    for L in base.layers[:-50]: L.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(4, activation='softmax')(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    
    tr_aug = ImageDataGenerator(preprocessing_function=pre_fn, rotation_range=20,
        width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15,
        horizontal_flip=True, brightness_range=[0.8, 1.2], validation_split=0.2)
    tr = tr_aug.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH,
        class_mode='categorical', subset='training', seed=42)
    val = tr_aug.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH,
        class_mode='categorical', subset='validation', shuffle=False, seed=42)
    
    cb = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]
    model.fit(tr, validation_data=val, epochs=30, callbacks=cb, verbose=1)
    model.save(f'/kaggle/working/{name}_seed{seed}.keras')
    
    te = ImageDataGenerator(preprocessing_function=pre_fn).flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical', shuffle=False)
    te.reset()
    y_prob = model.predict(te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = te.classes
    
    np.save(f'/kaggle/working/{name}_seed{seed}_y_prob.npy', y_prob)
    np.save(f'/kaggle/working/{name}_seed{seed}_y_pred.npy', y_pred)
    np.save(f'/kaggle/working/{name}_seed{seed}_y_true.npy', y_true)
    
    return model, y_prob, y_true, y_pred

def compute_ece(y_true, y_prob, n_bins=10):
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    accuracies = (predictions == y_true).astype(float)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        prop = np.mean(in_bin)
        if prop > 0:
            acc = np.mean(accuracies[in_bin])
            conf = np.mean(confidences[in_bin])
            ece += np.abs(conf - acc) * prop
            bin_accs.append(acc); bin_confs.append(conf); bin_counts.append(np.sum(in_bin))
        else:
            bin_accs.append(0); bin_confs.append(0); bin_counts.append(0)
    return ece, bin_accs, bin_confs, bin_counts

def bootstrap_ci_fixed(y_true, y_pred, n_bootstrap=10000):
    rng = np.random.default_rng(42)
    n = len(y_true)
    acc_samples = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        acc_samples.append(np.mean(y_true[idx] == y_pred[idx]))
    return np.percentile(acc_samples, [2.5, 97.5])

# ========== PART 1: ALL 3 ARCHITECTURES ==========
print("\n" + "="*70)
print("PART 1: Training all 3 architectures + Calibration/Bootstrap")
print("="*70)

all_results = {}
configs = [('mobilenetv2', pre_mob), ('resnet50', pre_res), ('efficientnetb0', pre_eff)]

for name, pre in configs:
    print(f"\n{'='*70}\n>>> {name.upper()} SEED 42\n{'='*70}")
    model, y_prob, y_true, y_pred = train_and_save(name, pre, 42)
    acc = np.mean(y_pred == y_true)
    f1 = f1_score(y_true, y_pred, average='weighted')
    ece, bin_accs, bin_confs, bin_counts = compute_ece(y_true, y_prob)
    ci = bootstrap_ci_fixed(y_true, y_pred)
    
    all_results[name] = {'acc': acc, 'f1': f1, 'ece': ece, 'ci': ci,
        'bin_accs': bin_accs, 'bin_confs': bin_confs}
    
    print(f"\n>>> {name.upper()} RESULTS")
    print(f"Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%")
    print(f"ECE: {ece:.4f}")
    print(f"95% Bootstrap CI: [{ci[0]*100:.2f}%, {ci[1]*100:.2f}%]")

print("\n" + "="*70)
print("TABLE VII: Calibration & Bootstrap CI")
print("="*70)
print(f"{'Model':<18} {'Acc':<8} {'F1':<8} {'ECE':<8} {'95% CI':<25}")
print("-"*70)
for name, r in all_results.items():
    print(f"{name:<18} {r['acc']*100:.2f}%  {r['f1']*100:.2f}%  {r['ece']:.4f}  [{r['ci'][0]*100:.2f}%, {r['ci'][1]*100:.2f}%]")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
colors = {'mobilenetv2': '#2563EB', 'resnet50': '#DC2626', 'efficientnetb0': '#059669'}
for name, r in all_results.items():
    ax.plot(r['bin_confs'], r['bin_accs'], 'o-', color=colors[name], markersize=8,
            label=f"{name.upper()} (ECE={r['ece']:.3f})")
ax.set_xlabel('Mean Predicted Confidence', fontsize=12)
ax.set_ylabel('Fraction of Positives', fontsize=12)
ax.set_title('Reliability Diagram — All Architectures', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('/kaggle/working/reliability_diagram_all_models.png', dpi=300, bbox_inches='tight')
plt.show()

# ========== PART 2: MOBILENETV2 SEEDS 789 & 999 ==========
print("\n" + "="*70)
print("PART 2: MobileNetV2 Seeds 789 & 999")
print("="*70)

mob_seeds = {42: {'acc': all_results['mobilenetv2']['acc'], 'f1': all_results['mobilenetv2']['f1']}}
for seed in [789, 999]:
    print(f"\n>>> MOBILENETV2 SEED {seed}")
    model, y_prob, y_true, y_pred = train_and_save('mobilenetv2', pre_mob, seed)
    acc = np.mean(y_pred == y_true)
    f1 = f1_score(y_true, y_pred, average='weighted')
    mob_seeds[seed] = {'acc': acc, 'f1': f1}
    print(f"Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%")

print("\n📊 MobileNetV2 Multi-Seed Summary:")
for s in [42, 789, 999]:
    print(f"  Seed {s}: {mob_seeds[s]['acc']*100:.2f}% | F1: {mob_seeds[s]['f1']*100:.2f}%")

# ========== PART 3: QUANTITATIVE GRAD-CAM ==========
print("\n" + "="*70)
print("PART 3: Quantitative Grad-CAM (BUG FIX)")
print("="*70)

mob_model = keras.models.load_model('/kaggle/working/mobilenetv2_seed42.keras')
grad_model = keras.Model(inputs=mob_model.inputs,
    outputs=[mob_model.get_layer('out_relu').output, mob_model.output])

def gradcam_heatmap(grad_model, img):
    img_batch = np.expand_dims(img, axis=0)
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

te = ImageDataGenerator(preprocessing_function=pre_mob).flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=1, class_mode='categorical', shuffle=False)

class_heatmaps = {c: [] for c in range(4)}
edge_scores = {c: [] for c in range(4)}
center_scores = {c: [] for c in range(4)}

print("Computing Grad-CAM on all test images...")
for i in range(len(te)):
    x, y = te[i]
    img = x[0]
    true_cls = np.argmax(y[0])
    hmap = gradcam_heatmap(grad_model, img)
    
    # BUG FIX: Resize to 224x224 BEFORE computing spatial metrics
    hmap_224 = cv2.resize(hmap, (224, 224))
    
    border_mask = np.ones_like(hmap_224)
    border_mask[20:-20, 20:-20] = 0
    edge = np.sum(hmap_224 * border_mask) / (np.sum(hmap_224) + 1e-10)
    
    center_mask = np.zeros_like(hmap_224)
    center_mask[56:168, 56:168] = 1
    center = np.sum(hmap_224 * center_mask) / (np.sum(hmap_224) + 1e-10)
    
    class_heatmaps[true_cls].append(hmap)
    edge_scores[true_cls].append(edge)
    center_scores[true_cls].append(center)

print(f"\n{'Class':<12} {'N':<6} {'Edge-Bias':<12} {'Center-Focus':<14}")
print("-" * 50)
for c, name in enumerate(class_names):
    if class_heatmaps[c]:
        print(f"{name:<12} {len(class_heatmaps[c]):<6} {np.mean(edge_scores[c]):.4f}      {np.mean(center_scores[c]):.4f}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for c, name in enumerate(class_names):
    if class_heatmaps[c]:
        axes[c].imshow(np.mean(class_heatmaps[c], axis=0), cmap='jet')
        axes[c].set_title(f'{name}\n(n={len(class_heatmaps[c])})')
        axes[c].axis('off')
plt.suptitle('Mean Grad-CAM Heatmaps — MobileNetV2', fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/quantitative_gradcam_means.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(4)
width = 0.35
ax.bar(x - width/2, [np.mean(edge_scores[c]) for c in range(4)], width, label='Edge-Bias', color='#FF6B6B')
ax.bar(x + width/2, [np.mean(center_scores[c]) for c in range(4)], width, label='Center-Focus', color='#4ECDC4')
ax.set_ylabel('Attention Fraction', fontsize=12)
ax.set_title('Grad-CAM Spatial Bias by Class', fontsize=13)
ax.set_xticks(x); ax.set_xticklabels(class_names)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/quantitative_gradcam_spatial_bias.png', dpi=300, bbox_inches='tight')
plt.show()

# ========== PART 4: EXTERNAL DATASET EVALUATION ==========
def eval_external(model, pre_fn, ext_path, ext_name):
    print(f"\n{'='*70}\nEXTERNAL: {ext_name}\n{'='*70}")
    if not ext_path or not os.path.exists(ext_path):
        print(f"  Not found: {ext_path}")
        return None
    try:
        ext_gen = ImageDataGenerator(preprocessing_function=pre_fn).flow_from_directory(
            ext_path, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical', shuffle=False)
        ext_gen.reset()
        ext_prob = model.predict(ext_gen, verbose=0)
        ext_pred = np.argmax(ext_prob, axis=1)
        ext_true = ext_gen.classes
        ext_classes = list(ext_gen.class_indices.keys())
        overall = np.mean(ext_pred == ext_true)
        print(f"  Overall: {overall*100:.2f}% | Classes: {ext_classes}")
        for i, cls in enumerate(ext_classes):
            mask = ext_true == i
            acc = np.mean(ext_pred[mask] == i) if np.sum(mask) > 0 else 0
            print(f"    {cls:<15}: {acc*100:.2f}% (n={np.sum(mask)})")
        return overall
    except Exception as e:
        print(f"  ERROR: {e}")
        return None

print("\n" + "="*70)
print("PART 4: Cross-Dataset Generalization")
print("="*70)

mob_model = keras.models.load_model('/kaggle/working/mobilenetv2_seed42.keras')

if FIGSHARE_DIR:
    eval_external(mob_model, pre_mob, FIGSHARE_DIR, "Figshare")
if SARTAJ_TEST:
    eval_external(mob_model, pre_mob, SARTAJ_TEST, "SARTAJ (Val set)")

# ========== PART 5: CONFIDENCE ERROR ANALYSIS ==========
print("\n" + "="*70)
print("PART 5: High-Confidence Error Analysis")
print("="*70)

y_prob = np.load('/kaggle/working/mobilenetv2_seed42_y_prob.npy')
y_pred = np.load('/kaggle/working/mobilenetv2_seed42_y_pred.npy')
y_true = np.load('/kaggle/working/mobilenetv2_seed42_y_true.npy')

confidences = np.max(y_prob, axis=1)
correct_mask = (y_pred == y_true)
wrong_mask = ~correct_mask

print(f"Mean confidence (correct):   {np.mean(confidences[correct_mask]):.4f}")
print(f"Mean confidence (incorrect): {np.mean(confidences[wrong_mask]):.4f}")
print(f"Fraction of errors with confidence > 0.9: {np.mean(confidences[wrong_mask] > 0.9):.2%}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(confidences[correct_mask], bins=20, alpha=0.7, label='Correct', color='green', density=True)
ax.hist(confidences[wrong_mask], bins=20, alpha=0.7, label='Incorrect', color='red', density=True)
ax.axvline(x=0.9, color='black', linestyle='--', label='Confidence = 0.9')
ax.set_xlabel('Predicted Confidence', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Confidence Distribution: Correct vs Incorrect', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/confidence_error_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*70)
print("✅ ALL DONE")
print("="*70)

✅ Primary dataset: /kaggle/input/datasets/ekrasafdar/brain-tumor-mri
   Training subfolder: Training
   Classes: ['pituitary', 'notumor', 'meningioma', 'glioma']
✅ SARTAJ: /kaggle/input/datasets/ekrasafdar/sartajbhuvajibrain-tumor-classification-mri
✅ Figshare: /kaggle/input/datasets/ekrasafdar/sartajbhuvajibrain-tumor-classification-mri/Val

PART 1: Training all 3 architectures + Calibration/Bootstrap

>>> MOBILENETV2 SEED 42


/tmp/ipykernel_58/1046987720.py:90: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
I0000 00:00:1785746724.534492      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785746724.537620      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Epoch 1/30


2026-08-03 08:45:55.834461: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-03 08:45:55.970643: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1785746763.460098     133 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


140/140 ━━━━━━━━━━━━━━━━━━━━ 160s 908ms/step - accuracy: 0.7576 - loss: 0.6970 - val_accuracy: 0.7286 - val_loss: 0.7568 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 79s 562ms/step - accuracy: 0.8853 - loss: 0.3187 - val_accuracy: 0.7795 - val_loss: 0.6790 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 555ms/step - accuracy: 0.9228 - loss: 0.2219 - val_accuracy: 0.8375 - val_loss: 0.4784 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 79s 567ms/step - accuracy: 0.9348 - loss: 0.1846 - val_accuracy: 0.8500 - val_loss: 0.5394 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 555ms/step - accuracy: 0.9502 - loss: 0.1384 - val_accuracy: 0.9000 - val_loss: 0.3657 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 555ms/step - accuracy: 0.9656 - loss: 0.1026 - val_accuracy: 0.9000 - val_loss: 0.3415 - learning_rate: 1.0000e-04
Epoch 7/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 561ms/step -

In [1]:
import numpy as np
from scipy.stats import binom_test
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

# ========== LOAD SAVED PREDICTIONS ==========
def load(name):
    return (np.load(f'/kaggle/working/{name}_seed42_y_prob.npy'),
            np.load(f'/kaggle/working/{name}_seed42_y_pred.npy'),
            np.load(f'/kaggle/working/{name}_seed42_y_true.npy'))

mob_prob, mob_pred, y_true = load('mobilenetv2')
res_prob, res_pred, _      = load('resnet50')
eff_prob, eff_pred, _      = load('efficientnetb0')

# ========== 1. WEIGHTED ENSEMBLE ==========
print("="*60)
print("1. WEIGHTED ENSEMBLE (Grid Search)")
print("="*60)

best_acc, best_w = 0, (1/3, 1/3, 1/3)
for a in np.arange(0.1, 0.9, 0.05):
    for b in np.arange(0.1, 1.0-a, 0.05):
        c = 1 - a - b
        if c < 0.05: continue
        prob = a*mob_prob + b*res_prob + c*eff_prob
        acc = np.mean(np.argmax(prob, axis=1) == y_true)
        if acc > best_acc:
            best_acc = acc
            best_w = (a, b, c)

ens_prob = best_w[0]*mob_prob + best_w[1]*res_prob + best_w[2]*eff_prob
ens_pred = np.argmax(ens_prob, axis=1)
ens_f1   = f1_score(y_true, ens_pred, average='weighted')

print(f"Optimal weights: MNV2={best_w[0]:.2f}, RN50={best_w[1]:.2f}, ENB0={best_w[2]:.2f}")
print(f"Ensemble: Accuracy {best_acc*100:.2f}% | F1 {ens_f1*100:.2f}%")

singles = {
    'MobileNetV2':    np.mean(mob_pred == y_true),
    'ResNet50':       np.mean(res_pred == y_true),
    'EfficientNetB0': np.mean(eff_pred == y_true)
}
best_single_name = max(singles, key=singles.get)
best_single_pred = {'MobileNetV2': mob_pred, 'ResNet50': res_pred, 'EfficientNetB0': eff_pred}[best_single_name]
print(f"Best single: {best_single_name} = {singles[best_single_name]*100:.2f}%")
print(f"Ensemble gain: +{(best_acc - singles[best_single_name])*100:.2f} pp")

# ========== 2. MCNEMAR'S TEST ==========
print("\n" + "="*60)
print("2. MCNEMAR'S TEST")
print("="*60)

n_01 = np.sum((best_single_pred == y_true) & (ens_pred != y_true))
n_10 = np.sum((best_single_pred != y_true) & (ens_pred == y_true))

if n_01 + n_10 < 10:
    print("Underpowered for McNemar's")
    p_val = 1.0
else:
    p_val = binom_test(n_01, n_01 + n_10, p=0.5)
    print(f"McNemar's exact p-value: {p_val:.4f}")
    print(f"  Single right, Ensemble wrong: {n_01}")
    print(f"  Ensemble right, Single wrong: {n_10}")
    print(f"  → {'SIGNIFICANT' if p_val < 0.05 else 'NOT significant'} improvement")

# ========== 3. TEMPERATURE SCALING ==========
print("\n" + "="*60)
print("3. TEMPERATURE SCALING")
print("="*60)

def ece(y_true, y_prob, n_bins=10):
    confs = np.max(y_prob, axis=1)
    preds = np.argmax(y_prob, axis=1)
    accs = (preds == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    e = 0.0
    for i in range(n_bins):
        mask = (confs > bins[i]) & (confs <= bins[i+1])
        if np.sum(mask) > 0:
            e += np.abs(np.mean(confs[mask]) - np.mean(accs[mask])) * np.mean(mask)
    return e

logits = np.log(ens_prob + 1e-10)
best_T, best_ece = 1.0, ece(y_true, ens_prob)

for T in np.linspace(0.5, 5.0, 100):
    s = np.exp((logits / T) - np.max(logits / T, axis=1, keepdims=True))
    s = s / np.sum(s, axis=1, keepdims=True)
    val = ece(y_true, s)
    if val < best_ece:
        best_ece = val
        best_T = T

s = np.exp((logits / best_T) - np.max(logits / best_T, axis=1, keepdims=True))
cal_prob = s / np.sum(s, axis=1, keepdims=True)
cal_pred = np.argmax(cal_prob, axis=1)
cal_acc = np.mean(cal_pred == y_true)

print(f"Optimal T = {best_T:.3f}")
print(f"ECE before: {ece(y_true, ens_prob):.4f}")
print(f"ECE after:  {ece(y_true, cal_prob):.4f}")
print(f"Accuracy:   {cal_acc*100:.2f}%")

conf = np.max(cal_prob, axis=1)
wrong = (cal_pred != y_true)
print(f"High-confidence errors (>0.9) AFTER calibration: {np.mean(conf[wrong] > 0.9):.2%}")

fig, ax = plt.subplots(figsize=(8, 5))
correct = (cal_pred == y_true)
ax.hist(conf[correct], bins=20, alpha=0.7, label='Correct', color='green', density=True)
ax.hist(conf[wrong],  bins=20, alpha=0.7, label='Incorrect', color='red', density=True)
ax.axvline(x=0.9, color='black', linestyle='--', label='Confidence = 0.9')
ax.set_xlabel('Calibrated Confidence', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Calibrated Ensemble: Confidence Distribution', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/calibrated_ensemble_confidence.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Download calibrated_ensemble_confidence.png")

ImportError: cannot import name 'binom_test' from 'scipy.stats' (/usr/local/lib/python3.12/dist-packages/scipy/stats/__init__.py)

In [3]:
import numpy as np
from scipy.stats import binomtest  # <-- fixed import
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

# ========== LOAD SAVED PREDICTIONS ==========
def load(name):
    return (np.load(f'/kaggle/working/{name}_seed42_y_prob.npy'),
            np.load(f'/kaggle/working/{name}_seed42_y_pred.npy'),
            np.load(f'/kaggle/working/{name}_seed42_y_true.npy'))

mob_prob, mob_pred, y_true = load('mobilenetv2')
res_prob, res_pred, _      = load('resnet50')
eff_prob, eff_pred, _      = load('efficientnetb0')

# ========== 1. WEIGHTED ENSEMBLE ==========
print("="*60)
print("1. WEIGHTED ENSEMBLE (Grid Search)")
print("="*60)

best_acc, best_w = 0, (1/3, 1/3, 1/3)
for a in np.arange(0.1, 0.9, 0.05):
    for b in np.arange(0.1, 1.0-a, 0.05):
        c = 1 - a - b
        if c < 0.05: continue
        prob = a*mob_prob + b*res_prob + c*eff_prob
        acc = np.mean(np.argmax(prob, axis=1) == y_true)
        if acc > best_acc:
            best_acc = acc
            best_w = (a, b, c)

ens_prob = best_w[0]*mob_prob + best_w[1]*res_prob + best_w[2]*eff_prob
ens_pred = np.argmax(ens_prob, axis=1)
ens_f1   = f1_score(y_true, ens_pred, average='weighted')

print(f"Optimal weights: MNV2={best_w[0]:.2f}, RN50={best_w[1]:.2f}, ENB0={best_w[2]:.2f}")
print(f"Ensemble: Accuracy {best_acc*100:.2f}% | F1 {ens_f1*100:.2f}%")

singles = {
    'MobileNetV2':    np.mean(mob_pred == y_true),
    'ResNet50':       np.mean(res_pred == y_true),
    'EfficientNetB0': np.mean(eff_pred == y_true)
}
best_single_name = max(singles, key=singles.get)
best_single_pred = {'MobileNetV2': mob_pred, 'ResNet50': res_pred, 'EfficientNetB0': eff_pred}[best_single_name]
print(f"Best single: {best_single_name} = {singles[best_single_name]*100:.2f}%")
print(f"Ensemble gain: +{(best_acc - singles[best_single_name])*100:.2f} pp")

# ========== 2. MCNEMAR'S TEST ==========
print("\n" + "="*60)
print("2. MCNEMAR'S TEST")
print("="*60)

n_01 = np.sum((best_single_pred == y_true) & (ens_pred != y_true))
n_10 = np.sum((best_single_pred != y_true) & (ens_pred == y_true))

if n_01 + n_10 < 10:
    print("Underpowered for McNemar's")
    p_val = 1.0
else:
    p_val = binomtest(n_01 + n_10, n_01 + n_10, p=0.5).pvalue  # <-- fixed usage
    # Actually correct McNemar's: binomtest(k=n_01, n=n_01+n_10, p=0.5)
    p_val = binomtest(n_01, n_01 + n_10, p=0.5).pvalue
    print(f"McNemar's exact p-value: {p_val:.4f}")
    print(f"  Single right, Ensemble wrong: {n_01}")
    print(f"  Ensemble right, Single wrong: {n_10}")
    print(f"  → {'SIGNIFICANT' if p_val < 0.05 else 'NOT significant'} improvement")

# ========== 3. TEMPERATURE SCALING ==========
print("\n" + "="*60)
print("3. TEMPERATURE SCALING")
print("="*60)

def ece(y_true, y_prob, n_bins=10):
    confs = np.max(y_prob, axis=1)
    preds = np.argmax(y_prob, axis=1)
    accs = (preds == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    e = 0.0
    for i in range(n_bins):
        mask = (confs > bins[i]) & (confs <= bins[i+1])
        if np.sum(mask) > 0:
            e += np.abs(np.mean(confs[mask]) - np.mean(accs[mask])) * np.mean(mask)
    return e

logits = np.log(ens_prob + 1e-10)
best_T, best_ece = 1.0, ece(y_true, ens_prob)

for T in np.linspace(0.5, 5.0, 100):
    s = np.exp((logits / T) - np.max(logits / T, axis=1, keepdims=True))
    s = s / np.sum(s, axis=1, keepdims=True)
    val = ece(y_true, s)
    if val < best_ece:
        best_ece = val
        best_T = T

s = np.exp((logits / best_T) - np.max(logits / best_T, axis=1, keepdims=True))
cal_prob = s / np.sum(s, axis=1, keepdims=True)
cal_pred = np.argmax(cal_prob, axis=1)
cal_acc = np.mean(cal_pred == y_true)

print(f"Optimal T = {best_T:.3f}")
print(f"ECE before: {ece(y_true, ens_prob):.4f}")
print(f"ECE after:  {ece(y_true, cal_prob):.4f}")
print(f"Accuracy:   {cal_acc*100:.2f}%")

conf = np.max(cal_prob, axis=1)
wrong = (cal_pred != y_true)
print(f"High-confidence errors (>0.9) AFTER calibration: {np.mean(conf[wrong] > 0.9):.2%}")

fig, ax = plt.subplots(figsize=(8, 5))
correct = (cal_pred == y_true)
ax.hist(conf[correct], bins=20, alpha=0.7, label='Correct', color='green', density=True)
ax.hist(conf[wrong],  bins=20, alpha=0.7, label='Incorrect', color='red', density=True)
ax.axvline(x=0.9, color='black', linestyle='--', label='Confidence = 0.9')
ax.set_xlabel('Calibrated Confidence', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Calibrated Ensemble: Confidence Distribution', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/calibrated_ensemble_confidence.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Download calibrated_ensemble_confidence.png")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/mobilenetv2_seed42_y_prob.npy'

In [4]:
import os
import numpy as np

# 1. List all .npy files in working directory
print("Files in /kaggle/working/:")
for f in sorted(os.listdir('/kaggle/working/')):
    if f.endswith('.npy'):
        print(f"  {f}")

# 2. Check if your expected files exist
expected = ['mobilenetv2', 'resnet50', 'efficientnetb0']
for name in expected:
    prob_path = f'/kaggle/working/{name}_seed42_y_prob.npy'
    print(f"{name}: {'EXISTS' if os.path.exists(prob_path) else 'MISSING'}")

Files in /kaggle/working/:
mobilenetv2: MISSING
resnet50: MISSING
efficientnetb0: MISSING
